# Laboratorio 01 — Parametrización con Widgets sobre tu propio dataset

**Semana:** 04 | **Actividad de referencia:** Actividad 01  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica los widgets de Databricks de la Actividad 01 para parametrizar el análisis de **tu propio dataset**. El objetivo es que un analista pueda cambiar el dataset consultado, las columnas de análisis y los filtros sin tocar el código.

## Parte 1 — Descripción del dataset

1. **Nombre, fuente y URL** del dataset.
2. **Parámetros que tendrá tu notebook:** ¿Qué cosas debería poder configurar el usuario sin modificar código? (filtros, columnas, umbrales de calidad, etc.)
3. **Preguntas de negocio:** ¿Qué análisis hará el notebook después de recibir los parámetros?

**Escribe tu respuesta aquí:**

## Parte 2 — Cargar el dataset como tabla Delta

In [ ]:
VOL     = "/Volumes/workspace/default/week_4"  # ajusta si usas otra ubicación
ARCHIVO = "tu_archivo.csv"
TABLA   = "workspace.default.lab04_01_mi_dataset"

df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{VOL}/{ARCHIVO}")

df.write.format("delta").mode("overwrite").saveAsTable(TABLA)
print(f"✓ {TABLA}: {df.count():,} filas x {len(df.columns)} columnas")
df.printSchema()

## Parte 3 — Perfil técnico del dataset

In [ ]:
from pyspark.sql import functions as F

# Tipos de datos disponibles para config de widgets
tipos_columnas = {
    "strings":   [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"],
    "numericas": [f.name for f in df.schema.fields if str(f.dataType) in ("DoubleType()", "IntegerType()", "LongType()", "FloatType()")]
}
print("Columnas string (buenas para filtros dropdown):", tipos_columnas["strings"])
print("Columnas numéricas (buenas para umbrales):    ", tipos_columnas["numericas"])

In [ ]:
# Valores únicos para alimentar los widgets dropdown
# Reemplaza 'columna_categoria' con el nombre real de la columna
valores_categoria = [row[0] for row in df.select("columna_categoria").distinct().orderBy("columna_categoria").collect()]
print("Valores únicos para widget dropdown:", valores_categoria)

## Parte 4 — Definir Widgets con dbutils

Define todos los widgets antes de usarlos. Implementa al menos uno de cada tipo.

In [ ]:
# Limpiar widgets anteriores para evitar errores si se re-ejecuta el notebook
dbutils.widgets.removeAll()

# Widget tipo TEXT: valor numérico libre como umbral
dbutils.widgets.text(
    name        = "umbral_minimo",
    defaultValue= "0",
    label       = "Umbral mínimo de columna numérica"
)

# Widget tipo DROPDOWN: selección única de categoría
# Usa los valores reales de tu dataset
dbutils.widgets.dropdown(
    name         = "categoria_filtro",
    defaultValue = valores_categoria[0] if valores_categoria else "",
    choices      = valores_categoria,
    label        = "Filtrar por categoría"
)

# Widget tipo COMBOBOX: columna numérica a analizar (puede escribirse o elegirse)
dbutils.widgets.combobox(
    name         = "columna_analisis",
    defaultValue = tipos_columnas["numericas"][0] if tipos_columnas["numericas"] else "",
    choices      = tipos_columnas["numericas"],
    label        = "Columna numérica a analizar"
)

# Widget tipo MULTISELECT: columnas a mostrar en el resultado final
dbutils.widgets.multiselect(
    name         = "columnas_salida",
    defaultValue = df.columns[0],
    choices      = df.columns,
    label        = "Columnas a mostrar en el resultado"
)

print("✓ Widgets definidos")

## Parte 5 — Leer valores de los widgets y ejecutar análisis

In [ ]:
# Leer los valores de los widgets
umbral         = float(dbutils.widgets.get("umbral_minimo"))
cat_filtro     = dbutils.widgets.get("categoria_filtro")
col_analisis   = dbutils.widgets.get("columna_analisis")
cols_salida    = dbutils.widgets.get("columnas_salida").split(",")

print(f"Parámetros recibidos:")
print(f"  umbral_minimo  = {umbral}")
print(f"  categoria      = {cat_filtro}")
print(f"  col_analisis   = {col_analisis}")
print(f"  cols_salida    = {cols_salida}")

In [ ]:
# Aplicar los parámetros al análisis
# Asegúrate de que 'columna_categoria' ya es el nombre real de la columna de filtro
df_filtrado = spark.table(TABLA) \
    .filter(F.col("columna_categoria") == cat_filtro) \
    .filter(F.col(col_analisis) >= umbral)

print(f"Filas después de filtros: {df_filtrado.count():,}")

df_filtrado.select(cols_salida) \
    .orderBy(F.col(col_analisis).desc()) \
    .show(20, truncate=False)

In [ ]:
# Estadísticas de la columna seleccionada con el filtro aplicado
df_filtrado.select(col_analisis).summary("count", "mean", "stddev", "min", "25%", "75%", "max").show()

**Análisis:**  
Cambia los valores de los widgets y vuelve a ejecutar el notebook desde la Parte 5. Responde:
1. ¿Cómo cambian las estadísticas al seleccionar diferentes categorías?
2. ¿Qué umbral `umbral_minimo` deja fuera más del 20% de los registros?
3. ¿El widget `combobox` te permite escribir un nombre de columna que no existe? ¿Qué error produce?

## Parte 6 — Encadenar con dbutils.notebook

Simula cómo este notebook sería llamado desde un notebook orquestador.

In [ ]:
# En el notebook orquestador se usaría:
# result = dbutils.notebook.run(
#     path      = "/ruta/a/este/notebook",
#     timeout_seconds = 300,
#     arguments = {
#         "umbral_minimo":   "100",
#         "categoria_filtro": "categoria_A",
#         "columna_analisis": "columna_numerica",
#         "columnas_salida":  "col1,col2,col3"
#     }
# )
# print(result)

# Devolver resultado al notebook padre con notebook.exit
resultado_kpi = df_filtrado.agg(F.avg(col_analisis)).collect()[0][0]
dbutils.notebook.exit(str(round(resultado_kpi, 4)))

# Nota: la celda se detendrá aquí al ejecutar. Comenta la línea exit() para seguir ejecutando.

## Parte 7 — Reflexión final

1. ¿Qué tipo de widget es más apropiado para cada caso: texto libre, dropdown, combobox o multiselect?
2. ¿Por qué es importante llamar `dbutils.widgets.removeAll()` al inicio de un notebook parametrizado?
3. ¿Cómo cambiaría el diseño si el notebook necesitara manejar 10 fuentes de datos distintas en lugar de 2 categorías?
4. ¿Qué riesgo de seguridad existe si se usa `dbutils.widgets.get()` directamente en una consulta SQL sin validación?

---

## Entrega en Git

```bash
git add semana_04/laboratorios/lab_01_widgets.ipynb
git commit -m "lab: semana04 lab01 widgets parametrizacion <nombre-dataset> - <tu-nombre>"
git push origin feature/semana04-metadata-<tu-nombre>
```